In [1]:
# 6-17-2026

In [2]:
import xarray as xr

In [3]:
zarr_path = "seasfire_filtered.zarr"

In [4]:
ds_filtered = xr.open_zarr(zarr_path, consolidated=True)

In [5]:
ds_filtered

<xarray.Dataset> Size: 76GB
Dimensions:                         (latitude: 720, longitude: 1440, time: 506)
Coordinates:
  * latitude                        (latitude) float64 6kB 89.88 ... -89.88
  * longitude                       (longitude) float64 12kB -179.9 ... 179.9
  * time                            (time) datetime64[ns] 4kB 2011-01-01 ... ...
Data variables: (12/41)
    area                            (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    biomes                          (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    cams_co2fire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    cams_frpfire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_max                (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_mean               (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ...                              ...
    t2m_max                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_mean                        (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_min                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    tp                              (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    vpd                             (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ws10                            (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
Attributes:
    crs:          EPSG:4326
    description:  The SeasFire Cube is a scientific datacube for seasonal fir...
    title:        SeasFire Cube: A Global Dataset for Seasonal Fire Modeling ...

In [6]:
from dask.diagnostics import ProgressBar

In [7]:
nonzero_count_delayed = (ds_filtered["gwis_ba"] > 0).sum()

In [8]:
with ProgressBar():
    total_nonzero = nonzero_count_delayed.compute()

[########################################] | 100% Completed | 1.78 sms


In [9]:
total_nonzero

<xarray.DataArray 'gwis_ba' ()> Size: 8B
array(3590844)

In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [13]:
features_to_exclude = ["fcci_ba_valid_mask", "fcci_fraction_of_observed_area", "fcci_fraction_of_burnable_area", "fcci_number_of_patches"]
features_to_include = [v for v in ds_filtered.data_vars if v not in features_to_exclude]

In [14]:
fire_mask = (ds_filtered["gwis_ba"].notnull()) & (ds_filtered["gwis_ba"] > 0)

In [15]:
time_idx, lat_idx, lon_idx = np.where(fire_mask.values)

In [16]:
data_dict = {
    "time": ds_filtered.time.values[time_idx],
    "latitude": ds_filtered.latitude.values[lat_idx],
    "longitude": ds_filtered.longitude.values[lon_idx],
    "gwis_ba_target": ds_filtered["gwis_ba"].values[time_idx, lat_idx, lon_idx]
}

In [17]:
for var in tqdm(features_to_include):
    dims = ds_filtered[var].dims
    var_data = ds_filtered[var].values
    
    if dims == ("time", "latitude", "longitude"):
        data_dict[var] = var_data[time_idx, lat_idx, lon_idx]
    elif dims == ("latitude", "longitude"):
        data_dict[var] = var_data[lat_idx, lon_idx]
    elif dims == ("time",):
        data_dict[var] = var_data[time_idx]

100%|██████████| 37/37 [02:29<00:00,  4.05s/it]


In [18]:
df = pd.DataFrame(data_dict) # build df

In [ ]:
df.to_csv("fire_events.csv", index=False)

In [1]:
import pandas as pd

In [ ]:
df_read = pd.read_csv("fire_events.csv")